# Modulo 8: Tecnicas de Visualizacao de Dados

**Disciplina:** Mineracao de Dados | **Semestre:** 8o | **Curso:** Engenharia de Software | **Instituicao:** UEPA/CCNT

---

### Objetivos de Aprendizagem
Ao final deste notebook voce sera capaz de:
- Entender a importancia da visualizacao atraves do Quarteto de Anscombe
- Criar 12+ tipos de graficos estatisticos com Matplotlib e Seaborn
- Aplicar PCA, t-SNE e UMAP para visualizacao de alta dimensao
- Construir visualizacoes interativas e animadas com Plotly
- Visualizar e interpretar resultados de modelos de ML (ROC, SHAP, importancia)
- Visualizar resultados de clustering com silhouette, elbow e radar charts
- Visualizar regras de associacao com grafos e heatmaps
- Gerar relatorios automaticos de EDA com ydata-profiling
- Construir um dashboard analitico completo com Plotly subplots

---

### Sumario
1. Quarteto de Anscombe - Por que Visualizar?
2. Graficos Estatisticos Fundamentais
3. Visualizacao de Alta Dimensao (PCA, t-SNE, UMAP)
4. Visualizacao Interativa com Plotly
5. Visualizacao de Resultados de ML
6. Visualizacao de Clustering
7. Visualizacao de Regras de Associacao
8. Auto-EDA com ydata-profiling
9. Mini-Dashboard com Plotly Subplots
10. Exercicios

In [ ]:
# Instalacao das dependencias
!pip install -q matplotlib seaborn plotly dash ydata-profiling sweetviz scikit-learn pandas numpy scipy shap umap-learn networkx wordcloud

## Setup - Imports e Configuracoes Globais

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
np.random.seed(42)

# Estilo matplotlib profissional
plt.rcParams.update({
    'figure.dpi': 100,
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3
})
sns.set_theme(style='whitegrid', palette='colorblind')

# Paleta de cores acessivel (colorblind-safe)
PALETTE_CB = ['#0072B2', '#E69F00', '#56B4E9', '#009E73',
              '#F0E442', '#D55E00', '#CC79A7', '#000000']
MINEBLUE = '#0072B2'
MINEORANGE = '#E69F00'
MINEGREEN = '#009E73'
MINERED = '#D55E00'

# Scikit-learn datasets
from sklearn.datasets import (load_iris, load_breast_cancer, load_digits,
                               fetch_california_housing, make_blobs)
from sklearn.preprocessing import StandardScaler

# Carregar datasets
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['species'] = [iris.target_names[t] for t in iris.target]

cancer = load_breast_cancer()
df_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df_cancer['target'] = cancer.target

print('Setup concluido!')
print(f'  Iris:           {df_iris.shape}')
print(f'  Breast Cancer:  {df_cancer.shape}')

## 1. Quarteto de Anscombe - Por que Visualizar?

Francis Anscombe (1973) criou 4 datasets com estatisticas resumidas identicas mas padroes completamente diferentes. Isso demonstra que **estatisticas nunca substituem a visualizacao**.

In [ ]:
# Quarteto de Anscombe - Dados exatos
anscombe = {
    'I':   {'x': [10,8,13,9,11,14,6,4,12,7,5],
             'y': [8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]},
    'II':  {'x': [10,8,13,9,11,14,6,4,12,7,5],
             'y': [9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]},
    'III': {'x': [10,8,13,9,11,14,6,4,12,7,5],
             'y': [7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]},
    'IV':  {'x': [8,8,8,8,8,8,8,19,8,8,8],
             'y': [6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89]}
}

# Calcular estatisticas
stats_table = []
for nome, dados in anscombe.items():
    x, y = np.array(dados['x']), np.array(dados['y'])
    n = len(x)
    # Regressao linear manual
    beta1 = np.cov(x, y, ddof=1)[0, 1] / np.var(x, ddof=1)
    beta0 = np.mean(y) - beta1 * np.mean(x)
    r = np.corrcoef(x, y)[0, 1]
    stats_table.append({
        'Dataset': nome,
        'Media X': round(np.mean(x), 2),
        'Media Y': round(np.mean(y), 2),
        'Var X': round(np.var(x, ddof=1), 2),
        'Var Y': round(np.var(y, ddof=1), 2),
        'Correlacao': round(r, 3),
        'Beta0': round(beta0, 3),
        'Beta1': round(beta1, 3)
    })

df_stats = pd.DataFrame(stats_table)
print('=== ESTATISTICAS DO QUARTETO DE ANSCOMBE ===')
print('(Todas as estatisticas sao praticamente identicas!)')
print(df_stats.to_string(index=False))

# Visualizar todos os 4 datasets
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
cores_anse = [MINEBLUE, MINEORANGE, MINEGREEN, MINERED]

for ax, (nome, dados), cor in zip(axes.flat, anscombe.items(), cores_anse):
    x, y = np.array(dados['x']), np.array(dados['y'])
    ax.scatter(x, y, color=cor, s=80, alpha=0.9, zorder=3, edgecolors='white', linewidth=0.5)
    # Linha de regressao
    beta1 = np.cov(x, y, ddof=1)[0, 1] / np.var(x, ddof=1)
    beta0 = np.mean(y) - beta1 * np.mean(x)
    x_line = np.linspace(min(x) - 1, max(x) + 1, 100)
    ax.plot(x_line, beta0 + beta1 * x_line, 'k--', linewidth=1.5, alpha=0.7,
            label=f'y = {beta0:.2f} + {beta1:.2f}x')
    ax.set_title(f'Dataset {nome}\n'
                 f'x={np.mean(x):.2f}, y={np.mean(y):.2f}, r={np.corrcoef(x,y)[0,1]:.3f}',
                 fontsize=11)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.legend(fontsize=9)
    ax.set_xlim(2, 20)
    ax.set_ylim(2, 14)

plt.suptitle('Quarteto de Anscombe (1973)\nMesmas estatisticas, padroes completamente diferentes!',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('anscombe.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nLicao principal: sempre visualize seus dados antes de confiar em estatisticas!')

## 2. Graficos Estatisticos Fundamentais

Galeria completa de 12 tipos de graficos: distribuicao, relacao, comparacao e composicao.

In [ ]:
import scipy.stats as stats
import squarify  # treemap; se nao tiver: pip install squarify

# Dataset California Housing para exemplos ricos
california = fetch_california_housing(as_frame=True)
df_cal = california.frame.sample(1000, random_state=42).reset_index(drop=True)

fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# 1. Histograma com KDE
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df_cal['MedHouseVal'], bins=30, color=MINEBLUE, alpha=0.7,
         density=True, edgecolor='white')
df_cal['MedHouseVal'].plot.kde(ax=ax1, color=MINERED, linewidth=2)
ax1.set_title('1. Histograma + KDE')
ax1.set_xlabel('Valor Mediano ($100k)')

# 2. Boxplot
ax2 = fig.add_subplot(gs[0, 1])
df_box = pd.DataFrame({
    'Grupo A': np.random.normal(50, 10, 200),
    'Grupo B': np.random.normal(60, 15, 200),
    'Grupo C': np.random.normal(45, 8, 200)
})
ax2.boxplot([df_box[c] for c in df_box.columns], labels=df_box.columns,
            patch_artist=True,
            boxprops=dict(facecolor=MINEBLUE, alpha=0.6),
            medianprops=dict(color=MINERED, linewidth=2))
ax2.set_title('2. Boxplot por Grupo')

# 3. Violin Plot
ax3 = fig.add_subplot(gs[0, 2])
df_violin = df_iris.copy()
species_list = df_violin['species'].unique()
data_violin = [df_violin[df_violin['species'] == s]['petal length (cm)'].values
               for s in species_list]
parts = ax3.violinplot(data_violin, showmeans=True, showmedians=True)
for pc, col in zip(parts['bodies'], [MINEBLUE, MINEORANGE, MINEGREEN]):
    pc.set_facecolor(col)
    pc.set_alpha(0.7)
ax3.set_xticks([1, 2, 3])
ax3.set_xticklabels(species_list, fontsize=8)
ax3.set_title('3. Violin Plot - Iris')
ax3.set_ylabel('Comprimento da Petala (cm)')

# 4. Q-Q Plot
ax4 = fig.add_subplot(gs[0, 3])
dados_qq = df_cal['MedInc'].dropna()
osm, osr = stats.probplot(dados_qq, dist='norm')
ax4.scatter(osm[0], osm[1], s=3, color=MINEBLUE, alpha=0.5)
ax4.plot(osm[0], osr[1] * osm[0] + osr[0], 'r--', linewidth=2)
ax4.set_title('4. Q-Q Plot (MedInc vs Normal)')
ax4.set_xlabel('Quantis Teoricos')
ax4.set_ylabel('Quantis Observados')

# 5. Scatter Plot
ax5 = fig.add_subplot(gs[1, 0])
colors_sc = {sp: col for sp, col in zip(df_iris['species'].unique(), PALETTE_CB[:3])}
for sp in df_iris['species'].unique():
    mask = df_iris['species'] == sp
    ax5.scatter(df_iris[mask]['sepal length (cm)'],
                df_iris[mask]['petal length (cm)'],
                c=colors_sc[sp], label=sp, s=40, alpha=0.8)
ax5.set_title('5. Scatter Plot - Iris')
ax5.set_xlabel('Sepala (cm)')
ax5.set_ylabel('Petala (cm)')
ax5.legend(fontsize=7)

# 6. Heatmap de Correlacao
ax6 = fig.add_subplot(gs[1, 1])
corr = df_iris.drop('species', axis=1).corr()
mask_upper = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=ax6, vmin=-1, vmax=1, center=0, mask=mask_upper,
            cbar=False)
ax6.set_title('6. Heatmap de Correlacao')

# 7. Grafico de Barras Agrupado
ax7 = fig.add_subplot(gs[1, 2])
categorias = ['Q1', 'Q2', 'Q3', 'Q4']
vendas_a = [45, 62, 78, 55]
vendas_b = [38, 71, 65, 72]
x_pos = np.arange(len(categorias))
width = 0.35
ax7.bar(x_pos - width/2, vendas_a, width, label='Produto A',
        color=MINEBLUE, alpha=0.8)
ax7.bar(x_pos + width/2, vendas_b, width, label='Produto B',
        color=MINEORANGE, alpha=0.8)
ax7.set_xticks(x_pos)
ax7.set_xticklabels(categorias)
ax7.set_title('7. Barras Agrupadas')
ax7.set_ylabel('Vendas (unidades)')
ax7.legend()

# 8. Lollipop Chart
ax8 = fig.add_subplot(gs[1, 3])
features = ['Feature A', 'Feature B', 'Feature C', 'Feature D', 'Feature E']
importancias = [0.35, 0.25, 0.20, 0.12, 0.08]
y_pos = range(len(features))
ax8.hlines(y_pos, 0, importancias, colors=MINEBLUE, alpha=0.5, linewidth=2)
ax8.plot(importancias, y_pos, 'o', color=MINEBLUE, markersize=10)
ax8.set_yticks(y_pos)
ax8.set_yticklabels(features)
ax8.set_title('8. Lollipop - Importancia')
ax8.set_xlabel('Importancia Relativa')

# 9. Line Plot - Serie Temporal
ax9 = fig.add_subplot(gs[2, 0])
t = np.linspace(0, 10, 100)
treino = 0.5 + 0.45 * (1 - np.exp(-0.5 * t)) + np.random.normal(0, 0.02, 100)
valid = 0.4 + 0.35 * (1 - np.exp(-0.4 * t)) + np.random.normal(0, 0.03, 100)
ax9.plot(t, treino, color=MINEBLUE, label='Treino', linewidth=2)
ax9.plot(t, valid, color=MINEORANGE, label='Validacao', linewidth=2, linestyle='--')
ax9.fill_between(t, valid - 0.03, valid + 0.03, alpha=0.2, color=MINEORANGE)
ax9.set_title('9. Linha - Curvas de Aprendizado')
ax9.set_xlabel('Epoca')
ax9.set_ylabel('Acuracia')
ax9.legend()

# 10. Stacked Bar - Composicao
ax10 = fig.add_subplot(gs[2, 1])
regioes = ['Norte', 'Sul', 'Leste', 'Oeste']
cat1 = [30, 45, 25, 50]
cat2 = [40, 30, 35, 25]
cat3 = [30, 25, 40, 25]
x_st = np.arange(len(regioes))
ax10.bar(x_st, cat1, color=MINEBLUE, label='Cat 1', alpha=0.85)
ax10.bar(x_st, cat2, bottom=cat1, color=MINEORANGE, label='Cat 2', alpha=0.85)
ax10.bar(x_st, cat3, bottom=np.array(cat1)+np.array(cat2),
         color=MINEGREEN, label='Cat 3', alpha=0.85)
ax10.set_xticks(x_st)
ax10.set_xticklabels(regioes)
ax10.set_title('10. Barras Empilhadas')
ax10.legend(fontsize=8)

# 11. Hexbin
ax11 = fig.add_subplot(gs[2, 2])
x_hex = np.random.randn(2000)
y_hex = x_hex * 0.7 + np.random.randn(2000) * 0.7
hb = ax11.hexbin(x_hex, y_hex, gridsize=25, cmap='Blues', mincnt=1)
plt.colorbar(hb, ax=ax11, label='Contagem')
ax11.set_title('11. Hexbin (grande dataset)')
ax11.set_xlabel('X')
ax11.set_ylabel('Y')

# 12. Grafico de Pizza (uso moderado)
ax12 = fig.add_subplot(gs[2, 3])
label_pizza = ['Positivo', 'Negativo', 'Neutro']
tamanhos = [45, 35, 20]
explode = (0.05, 0.05, 0.05)
ax12.pie(tamanhos, labels=label_pizza, explode=explode, autopct='%1.1f%%',
         colors=[MINEGREEN, MINERED, MINEBLUE],
         startangle=90, shadow=False)
ax12.set_title('12. Pizza - Sentimentos')

plt.suptitle('Galeria de Graficos Estatisticos Fundamentais', fontsize=16,
             fontweight='bold', y=1.01)
plt.savefig('galeria_graficos.png', dpi=120, bbox_inches='tight')
plt.show()
print('Galeria de 12 tipos de graficos criada com sucesso!')

In [ ]:
# Seaborn avancado: pairplot, FacetGrid, clustermap

# 1. Pairplot com regressao
g = sns.pairplot(df_iris, hue='species', palette=PALETTE_CB[:3],
                 plot_kws={'alpha': 0.7, 's': 30},
                 diag_kind='kde')
g.fig.suptitle('Pairplot Iris - Todos os Pares de Features', y=1.02, fontsize=13)
plt.savefig('pairplot_iris.png', dpi=100, bbox_inches='tight')
plt.show()

# 2. Clustermap (heatmap hierarquico)
X_cl = StandardScaler().fit_transform(df_iris.drop('species', axis=1))
df_scaled = pd.DataFrame(X_cl, columns=df_iris.columns[:-1])
row_colors_map = {'setosa': MINEBLUE, 'versicolor': MINEORANGE, 'virginica': MINEGREEN}
row_colors = df_iris['species'].map(row_colors_map)

g2 = sns.clustermap(df_scaled, row_colors=row_colors,
                    cmap='RdYlBu_r', figsize=(8, 8),
                    row_cluster=True, col_cluster=True)
g2.fig.suptitle('Clustermap - Iris (features normalizadas)', y=1.01, fontsize=12)
plt.savefig('clustermap_iris.png', dpi=100, bbox_inches='tight')
plt.show()
print('Graficos Seaborn avancados criados!')

## 3. Visualizacao de Alta Dimensao

Dataset Digits (64 features) reduzido para 2D com PCA, t-SNE e UMAP. Comparamos a separabilidade visual das classes.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import time

# Carregar dataset Digits (n=1797, 64 features = pixels 8x8)
digits = load_digits()
X_dig = digits.data
y_dig = digits.target

# Normalizar
scaler_dig = StandardScaler()
X_dig_sc = scaler_dig.fit_transform(X_dig)

resultados_dim = {}

# PCA -> 2D
t0 = time.time()
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_dig_sc)
resultados_dim['PCA'] = {'X': X_pca, 'tempo': time.time() - t0}
print(f'PCA:   {resultados_dim["PCA"]["tempo"]:.2f}s | '
      f'Variancia explicada: {pca.explained_variance_ratio_.sum():.1%}')

# t-SNE (primeiro reduzir com PCA para acelerar)
X_pca50 = PCA(n_components=50, random_state=42).fit_transform(X_dig_sc)
t0 = time.time()
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X_pca50)
resultados_dim['t-SNE'] = {'X': X_tsne, 'tempo': time.time() - t0}
print(f't-SNE: {resultados_dim["t-SNE"]["tempo"]:.2f}s | Perplexidade: 30')

# UMAP
try:
    import umap
    t0 = time.time()
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X_pca50)
    resultados_dim['UMAP'] = {'X': X_umap, 'tempo': time.time() - t0}
    print(f'UMAP:  {resultados_dim["UMAP"]["tempo"]:.2f}s | n_neighbors=15, min_dist=0.1')
    usar_umap = True
except ImportError:
    print('UMAP nao disponivel. Instale: pip install umap-learn')
    usar_umap = False
    X_umap = X_pca  # fallback
    resultados_dim['UMAP (PCA fallback)'] = {'X': X_umap, 'tempo': 0}

# Visualizar os 3 metodos
n_cols = 3
fig, axes = plt.subplots(1, n_cols, figsize=(18, 6))
cores_dig = plt.cm.tab10(np.linspace(0, 1, 10))

for ax, (nome, res) in zip(axes, resultados_dim.items()):
    X_2d = res['X']
    for digito in range(10):
        mask = y_dig == digito
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=[cores_dig[digito]], label=str(digito),
                   s=8, alpha=0.7)
    ax.set_title(f'{nome}\n({res["tempo"]:.2f}s)', fontsize=12)
    ax.legend(title='Digito', fontsize=6, ncol=2, markerscale=2)
    ax.set_xlabel('Componente 1')
    ax.set_ylabel('Componente 2')

plt.suptitle('Reducao de Dimensionalidade - Dataset Digits (64D -> 2D)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dim_reduction.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# PCA Biplot: observacoes + loadings das features originais
pca_biplot = PCA(n_components=2, random_state=42)
X_pca_iris = pca_biplot.fit_transform(
    StandardScaler().fit_transform(df_iris.drop('species', axis=1))
)
loadings = pca_biplot.components_.T
feature_names = df_iris.columns[:-1].tolist()

fig_biplot, ax_bp = plt.subplots(figsize=(9, 7))
# Observacoes
for sp, col in zip(df_iris['species'].unique(), PALETTE_CB[:3]):
    mask = df_iris['species'] == sp
    ax_bp.scatter(X_pca_iris[mask, 0], X_pca_iris[mask, 1],
                  c=col, label=sp, s=30, alpha=0.7)
# Loadings (vetores)
scale = 3.5
for i, (feat, (lx, ly)) in enumerate(zip(feature_names, loadings)):
    ax_bp.arrow(0, 0, lx * scale, ly * scale,
                head_width=0.1, head_length=0.05,
                fc=MINERED, ec=MINERED, linewidth=2)
    ax_bp.text(lx * scale * 1.15, ly * scale * 1.1,
               feat.replace(' (cm)', ''), fontsize=9,
               ha='center', color=MINERED, fontweight='bold')

ax_bp.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax_bp.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
ax_bp.set_xlabel(f'PC1 ({pca_biplot.explained_variance_ratio_[0]:.1%} var.)')
ax_bp.set_ylabel(f'PC2 ({pca_biplot.explained_variance_ratio_[1]:.1%} var.)')
ax_bp.set_title('PCA Biplot - Iris: Observacoes + Loadings das Features')
ax_bp.legend()
plt.tight_layout()
plt.savefig('pca_biplot.png', dpi=150, bbox_inches='tight')
plt.show()

# Parallel Coordinates com Plotly
df_parallel = df_iris.copy()
df_parallel['species_num'] = pd.Categorical(df_parallel['species']).codes
fig_pc = px.parallel_coordinates(
    df_parallel,
    color='species_num',
    dimensions=['sepal length (cm)', 'sepal width (cm)',
                'petal length (cm)', 'petal width (cm)'],
    color_continuous_scale=px.colors.diverging.Tealrose,
    title='Coordenadas Paralelas - Dataset Iris'
)
fig_pc.update_layout(height=400)
fig_pc.show()
print('Biplot e Coordenadas Paralelas criados!')

## 4. Visualizacao Interativa com Plotly

Graficos interativos: serie temporal com seletores, scatter animado (estilo Gapminder), sunburst e SPLOM.

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Serie temporal sintetica de acoes
dates = pd.date_range('2020-01-01', '2024-06-01', freq='B')  # dias uteis
np.random.seed(42)
n_pts = len(dates)

# Simulacao de 3 acoes com caminho aleatorio
def gerar_acao(preco_inicial, vol=0.015, drift=0.0003):
    retornos = np.random.normal(drift, vol, n_pts)
    return preco_inicial * np.exp(np.cumsum(retornos))

df_stocks = pd.DataFrame({
    'Data': dates,
    'Acao_A': gerar_acao(100.0, vol=0.018),
    'Acao_B': gerar_acao(85.0, vol=0.012, drift=0.0005),
    'Acao_C': gerar_acao(120.0, vol=0.022, drift=0.0001)
})

# Grafico de linhas interativo com seletores de periodo
fig_ts = go.Figure()
for col, color in zip(['Acao_A', 'Acao_B', 'Acao_C'],
                      ['#0072B2', '#E69F00', '#009E73']):
    fig_ts.add_trace(go.Scatter(
        x=df_stocks['Data'],
        y=df_stocks[col],
        name=col,
        line=dict(color=color, width=1.5),
        hovertemplate=f'<b>{col}</b><br>Data: %{{x|%d/%m/%Y}}<br>Preco: R$ %{{y:.2f}}<extra></extra>'
    ))

fig_ts.update_layout(
    title='Precos de Acoes (Simulado) - 2020 a 2024',
    xaxis=dict(
        rangeselector=dict(buttons=[
            dict(count=1, label='1m', step='month', stepmode='backward'),
            dict(count=6, label='6m', step='month', stepmode='backward'),
            dict(count=1, label='1a', step='year', stepmode='backward'),
            dict(label='Tudo', step='all')
        ]),
        rangeslider=dict(visible=True),
        type='date'
    ),
    yaxis=dict(title='Preco (R$)'),
    hovermode='x unified',
    height=500
)
fig_ts.show()

In [ ]:
# Animated Bubble Chart - estilo Gapminder
np.random.seed(42)
anos = list(range(2000, 2025))
paises = ['Brasil', 'Argentina', 'Chile', 'Colombia', 'Peru',
          'Mexico', 'Venezuela', 'Equador', 'Bolivia', 'Uruguay']
regioes = ['America do Sul'] * 9 + ['America Central']
n_paises = len(paises)

registros = []
for i, pais in enumerate(paises):
    pib_inicial = np.random.uniform(3000, 15000)
    esp_inicial = np.random.uniform(65, 78)
    pop_inicial = np.random.uniform(5, 215)
    for j, ano in enumerate(anos):
        fator = 1 + np.random.normal(0.025, 0.015)
        registros.append({
            'pais': pais, 'ano': str(ano), 'regiao': regioes[i],
            'pib_per_capita': pib_inicial * (fator ** j),
            'expectativa_vida': min(85, esp_inicial + j * 0.25 + np.random.normal(0, 0.5)),
            'populacao': pop_inicial * (1.01 ** j)
        })

df_gap = pd.DataFrame(registros)

fig_bubble = px.scatter(
    df_gap, x='pib_per_capita', y='expectativa_vida',
    size='populacao', color='regiao',
    hover_name='pais', animation_frame='ano',
    size_max=60, log_x=True,
    title='Desenvolvimento Humano - America Latina (Simulado, estilo Gapminder)',
    labels={'pib_per_capita': 'PIB per capita (USD, log)',
            'expectativa_vida': 'Expectativa de Vida (anos)'},
    color_discrete_sequence=PALETTE_CB,
    range_x=[1000, 50000], range_y=[60, 90]
)
fig_bubble.update_layout(height=500)
fig_bubble.show()

# Sunburst - hierarquia
fig_sun = px.sunburst(
    data_frame=pd.DataFrame({
        'Nivel1': ['ML','ML','ML','ML','DL','DL','DL','Stats','Stats'],
        'Nivel2': ['Classif.','Classif.','Cluster','Regress.',
                   'CNN','RNN','Transformer','Bayes','Regress.'],
        'Nivel3': ['SVM','RF','K-Means','Lin.Reg.',
                   'ResNet','LSTM','BERT','Naive Bayes','Log.Reg.'],
        'valor': [15, 20, 18, 12, 22, 17, 25, 10, 14]
    }),
    path=['Nivel1', 'Nivel2', 'Nivel3'],
    values='valor',
    title='Hierarquia de Algoritmos de Mineracao de Dados',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_sun.update_layout(height=500)
fig_sun.show()

## 5. Visualizacao de Resultados de ML

Treinamos XGBoost no dataset de cancer de mama e criamos todas as visualizacoes de avaliacao e interpretabilidade.

In [ ]:
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (roc_curve, auc, confusion_matrix,
                               precision_recall_curve, classification_report)
try:
    from xgboost import XGBClassifier
    clf = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                        use_label_encoder=False, eval_metric='logloss',
                        random_state=42)
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    clf = GradientBoostingClassifier(n_estimators=100, max_depth=4,
                                      learning_rate=0.1, random_state=42)

X_c = cancer.data
y_c = cancer.target
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

clf.fit(X_tr_c, y_tr_c)
y_pred_c = clf.predict(X_te_c)
y_prob_c = clf.predict_proba(X_te_c)[:, 1]

print(classification_report(y_te_c, y_pred_c,
                              target_names=['Maligno','Benigno']))

# Curvas de aprendizado
train_sizes, train_scores, val_scores = learning_curve(
    clf, X_c, y_c, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 8),
    scoring='accuracy'
)

# Todas as visualizacoes em um painel 2x3
fig, axes = plt.subplots(2, 3, figsize=(16, 11))

# 1. ROC
fpr_c, tpr_c, _ = roc_curve(y_te_c, y_prob_c)
auc_c = auc(fpr_c, tpr_c)
axes[0,0].plot(fpr_c, tpr_c, color=MINEBLUE, lw=2.5, label=f'AUC = {auc_c:.3f}')
axes[0,0].fill_between(fpr_c, tpr_c, alpha=0.1, color=MINEBLUE)
axes[0,0].plot([0,1],[0,1],'k--', alpha=0.4)
axes[0,0].set_xlabel('False Positive Rate')
axes[0,0].set_ylabel('True Positive Rate')
axes[0,0].set_title('Curva ROC - Cancer de Mama')
axes[0,0].legend(loc='lower right')

# 2. Precision-Recall
prec_c, rec_c, _ = precision_recall_curve(y_te_c, y_prob_c)
pr_auc = auc(rec_c, prec_c)
axes[0,1].plot(rec_c, prec_c, color=MINEORANGE, lw=2.5, label=f'AP = {pr_auc:.3f}')
axes[0,1].fill_between(rec_c, prec_c, alpha=0.1, color=MINEORANGE)
axes[0,1].set_xlabel('Recall')
axes[0,1].set_ylabel('Precision')
axes[0,1].set_title('Curva Precision-Recall')
axes[0,1].legend()

# 3. Confusion Matrix
cm_c = confusion_matrix(y_te_c, y_pred_c)
cm_norm = cm_c.astype('float') / cm_c.sum(axis=1, keepdims=True)
im = axes[0,2].imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[0,2])
for ii in range(2):
    for jj in range(2):
        axes[0,2].text(jj, ii, f'{cm_c[ii,jj]}\n({cm_norm[ii,jj]:.1%})',
                       ha='center', va='center', fontsize=11,
                       color='white' if cm_norm[ii,jj] > 0.5 else 'black')
axes[0,2].set_xticks([0,1])
axes[0,2].set_yticks([0,1])
axes[0,2].set_xticklabels(['Maligno','Benigno'])
axes[0,2].set_yticklabels(['Maligno','Benigno'])
axes[0,2].set_xlabel('Predito')
axes[0,2].set_ylabel('Real')
axes[0,2].set_title('Matriz de Confusao (normalizada)')

# 4. Learning Curves
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)
axes[1,0].plot(train_sizes, train_mean, 'o-', color=MINEBLUE, label='Treino')
axes[1,0].fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.15, color=MINEBLUE)
axes[1,0].plot(train_sizes, val_mean, 'o--', color=MINEORANGE, label='Validacao (CV)')
axes[1,0].fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.15, color=MINEORANGE)
axes[1,0].set_xlabel('Tamanho do Conjunto de Treino')
axes[1,0].set_ylabel('Acuracia')
axes[1,0].set_title('Curvas de Aprendizado')
axes[1,0].legend()

# 5. Feature Importance
importances_c = clf.feature_importances_
feat_names_c = cancer.feature_names
top_n = 10
top_idx = np.argsort(importances_c)[-top_n:]
axes[1,1].barh(range(top_n), importances_c[top_idx], color=MINEBLUE, alpha=0.8)
axes[1,1].set_yticks(range(top_n))
axes[1,1].set_yticklabels([feat_names_c[i] for i in top_idx], fontsize=8)
axes[1,1].set_xlabel('Importancia')
axes[1,1].set_title(f'Top-{top_n} Features Importantes')

# 6. Threshold Analysis
thresholds = np.linspace(0.01, 0.99, 100)
precisions, recalls, f1s = [], [], []
from sklearn.metrics import f1_score, precision_score, recall_score
for th in thresholds:
    preds_th = (y_prob_c >= th).astype(int)
    precisions.append(precision_score(y_te_c, preds_th, zero_division=0))
    recalls.append(recall_score(y_te_c, preds_th, zero_division=0))
    f1s.append(f1_score(y_te_c, preds_th, zero_division=0))
best_th = thresholds[np.argmax(f1s)]
axes[1,2].plot(thresholds, precisions, color=MINEBLUE, label='Precisao')
axes[1,2].plot(thresholds, recalls, color=MINEORANGE, label='Recall')
axes[1,2].plot(thresholds, f1s, color=MINEGREEN, linewidth=2.5, label='F1-Score')
axes[1,2].axvline(x=best_th, color=MINERED, linestyle='--',
                  label=f'Melhor threshold={best_th:.2f}')
axes[1,2].set_xlabel('Threshold de Decisao')
axes[1,2].set_ylabel('Score')
axes[1,2].set_title('Analise de Threshold')
axes[1,2].legend(fontsize=8)

plt.suptitle('Visualizacao de Resultados de ML - Cancer de Mama (XGBoost)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ml_results.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Visualizacoes SHAP
try:
    import shap
    shap.initjs()

    # Calcular SHAP values
    explainer = shap.TreeExplainer(clf)
    X_te_df = pd.DataFrame(X_te_c, columns=cancer.feature_names)
    shap_values = explainer.shap_values(X_te_df)

    # Se shap_values e lista (binario), pegar classe 1
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    print('=== SHAP - Summary Plot (Beeswarm) ===')
    shap.summary_plot(sv, X_te_df, feature_names=cancer.feature_names,
                      max_display=15, show=True)

    print('=== SHAP - Bar Summary ===')
    shap.summary_plot(sv, X_te_df, plot_type='bar',
                      feature_names=cancer.feature_names,
                      max_display=15, show=True)

    print('=== SHAP - Waterfall para 3 predicoes individuais ===')
    base_val = explainer.expected_value
    if isinstance(base_val, list):
        base_val = base_val[1]

    for idx_ex, descricao in [(0, 'Exemplo 0'), (5, 'Exemplo 5'), (10, 'Exemplo 10')]:
        print(f'  {descricao}: Real={y_te_c[idx_ex]}, Predito={y_pred_c[idx_ex]}')
        expl = shap.Explanation(
            values=sv[idx_ex],
            base_values=base_val,
            data=X_te_df.iloc[idx_ex].values,
            feature_names=cancer.feature_names
        )
        shap.waterfall_plot(expl, max_display=10, show=True)

    print('SHAP visualizacoes criadas com sucesso!')
except ImportError:
    print('SHAP nao disponivel. Instale: pip install shap')
    print('Mostrando importancia de features como alternativa:')
    top_feats = pd.Series(importances_c, index=cancer.feature_names)
    top_feats.nlargest(15).plot(kind='barh', color=MINEBLUE, alpha=0.8)
    plt.title('Importancia de Features (alternativa ao SHAP)')
    plt.tight_layout()
    plt.show()

## 6. Visualizacao de Clustering

Aplicamos K-Means ao dataset Iris e criamos visualizacoes: t-SNE, silhouette, elbow e radar chart dos centroides.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.manifold import TSNE
import matplotlib.cm as cm

# Preparar dataset (Iris normalizado)
X_iris = StandardScaler().fit_transform(df_iris.drop('species', axis=1))
n_clusters = 4

# Elbow curve
inertias_iris = []
sil_scores = []
K_vals = range(2, 9)
for k in K_vals:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_iris)
    inertias_iris.append(km.inertia_)
    sil_scores.append(silhouette_score(X_iris, labels_k))

# K-Means final
km_final = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_iris)

# t-SNE 2D
X_tsne_iris = TSNE(n_components=2, perplexity=20, random_state=42).fit_transform(X_iris)

# Silhouette samples
sil_samples = silhouette_samples(X_iris, cluster_labels)
sil_avg = silhouette_score(X_iris, cluster_labels)

fig_cl = plt.figure(figsize=(18, 12))
gs_cl = gridspec.GridSpec(2, 3, figure=fig_cl, hspace=0.4, wspace=0.4)

# 1. t-SNE colorido por cluster
ax_ts = fig_cl.add_subplot(gs_cl[0, 0])
for k in range(n_clusters):
    mask_k = cluster_labels == k
    ax_ts.scatter(X_tsne_iris[mask_k, 0], X_tsne_iris[mask_k, 1],
                  c=PALETTE_CB[k], s=40, alpha=0.8, label=f'Cluster {k}')
ax_ts.set_title(f't-SNE Colorido por Cluster (k={n_clusters})')
ax_ts.legend(fontsize=9)
ax_ts.set_xlabel('t-SNE 1')
ax_ts.set_ylabel('t-SNE 2')

# 2. Elbow Curve
ax_el = fig_cl.add_subplot(gs_cl[0, 1])
ax_el2 = ax_el.twinx()
ax_el.plot(K_vals, inertias_iris, 'bo-', label='Inertia', linewidth=2)
ax_el2.plot(K_vals, sil_scores, 'rs--', label='Silhouette', linewidth=2)
ax_el.axvline(x=n_clusters, color='gray', linestyle=':', alpha=0.7)
ax_el.set_xlabel('Numero de Clusters k')
ax_el.set_ylabel('Inertia', color='blue')
ax_el2.set_ylabel('Silhouette Score', color='red')
ax_el.set_title('Curva do Cotovelo + Silhouette Score')
ax_el.legend(loc='upper right')
ax_el2.legend(loc='upper center')

# 3. Silhouette Plot
ax_sil = fig_cl.add_subplot(gs_cl[0, 2])
y_lower = 10
for k in range(n_clusters):
    ith_cluster_sil = sil_samples[cluster_labels == k]
    ith_cluster_sil.sort()
    size_cluster_i = ith_cluster_sil.shape[0]
    y_upper = y_lower + size_cluster_i
    color = PALETTE_CB[k]
    ax_sil.fill_betweenx(np.arange(y_lower, y_upper),
                          0, ith_cluster_sil,
                          facecolor=color, alpha=0.7)
    ax_sil.text(-0.05, y_lower + 0.5 * size_cluster_i, f'C{k}', fontsize=9)
    y_lower = y_upper + 10
ax_sil.axvline(x=sil_avg, color=MINERED, linestyle='--',
               label=f'Media={sil_avg:.3f}')
ax_sil.set_xlabel('Coeficiente de Silhouette')
ax_sil.set_title(f'Silhouette Plot (k={n_clusters})')
ax_sil.legend()

# 4. Radar Chart dos Centroides
ax_radar = fig_cl.add_subplot(gs_cl[1, :], polar=True)
feature_nms = df_iris.columns[:-1].tolist()
N = len(feature_nms)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
centroids_orig = pd.DataFrame(
    StandardScaler().fit(df_iris.drop('species', axis=1))
    .inverse_transform(km_final.cluster_centers_),
    columns=feature_nms
)
for k in range(n_clusters):
    values = centroids_orig.iloc[k].tolist()
    values += values[:1]
    ax_radar.plot(angles, values, linewidth=2, label=f'Cluster {k}',
                  color=PALETTE_CB[k])
    ax_radar.fill(angles, values, alpha=0.1, color=PALETTE_CB[k])
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(feature_nms, fontsize=9)
ax_radar.set_title(f'Radar Chart - Perfil dos {n_clusters} Centroides (Iris)', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.suptitle('Visualizacao de Clustering - K-Means (Iris)', fontsize=15, fontweight='bold')
plt.savefig('clustering_viz.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Visualizacao de Regras de Associacao

Geramos dados de mercado, aplicamos Apriori e visualizamos com grafo, heatmap e bubble chart.

In [ ]:
import networkx as nx
import itertools

# Dataset de mercado sintetico
np.random.seed(42)
produtos = ['Leite', 'Pao', 'Manteiga', 'Queijo', 'Cafe', 'Acucar',
            'Ovo', 'Iogurte', 'Presunto', 'Suco']

# Padroes de co-ocorrencia
basket_patterns = [
    ['Leite', 'Pao', 'Manteiga'],        # cafe da manha
    ['Leite', 'Cafe', 'Acucar'],          # bebidas
    ['Queijo', 'Presunto', 'Pao'],        # sanduiche
    ['Ovo', 'Manteiga', 'Cafe'],          # cafe com ovos
    ['Iogurte', 'Leite', 'Suco'],         # laticínios + bebida
    ['Pao', 'Manteiga', 'Queijo'],        # cafe simples
    ['Leite', 'Acucar', 'Cafe', 'Pao'],  # combo
    ['Ovo', 'Queijo', 'Presunto'],        # proteinas
]

# Gerar transacoes
transacoes = []
for _ in range(500):
    padrao = np.random.choice(len(basket_patterns))
    base = list(basket_patterns[padrao])
    if np.random.random() > 0.5:
        base.append(np.random.choice(produtos))
    transacoes.append(list(set(base)))

# Calcular suporte manualmente
n_trans = len(transacoes)
suporte_item = {p: sum(1 for t in transacoes if p in t) / n_trans for p in produtos}

# Co-ocorrencia de pares
co_ocorrencia = {}
for p1, p2 in itertools.combinations(produtos, 2):
    count = sum(1 for t in transacoes if p1 in t and p2 in t)
    co_ocorrencia[(p1, p2)] = count / n_trans

# Regras de associacao simples
min_suporte = 0.08
min_confianca = 0.4
regras = []
for (p1, p2), sup in co_ocorrencia.items():
    if sup >= min_suporte:
        conf12 = sup / suporte_item[p1]
        conf21 = sup / suporte_item[p2]
        lift12 = conf12 / suporte_item[p2]
        lift21 = conf21 / suporte_item[p1]
        if conf12 >= min_confianca:
            regras.append({'antecedente': p1, 'consequente': p2,
                           'suporte': sup, 'confianca': conf12, 'lift': lift12})
        if conf21 >= min_confianca:
            regras.append({'antecedente': p2, 'consequente': p1,
                           'suporte': sup, 'confianca': conf21, 'lift': lift21})

df_regras = pd.DataFrame(regras).drop_duplicates()
print(f'Regras encontradas: {len(df_regras)}')
print(df_regras.sort_values('lift', ascending=False).head(10).to_string(index=False))

# VISUALIZACOES
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# 1. Bubble Chart: suporte vs confianca, tamanho = lift
scatter = axes[0].scatter(
    df_regras['suporte'], df_regras['confianca'],
    s=df_regras['lift'] * 200,
    c=df_regras['lift'], cmap='YlOrRd',
    alpha=0.7, edgecolors='gray', linewidth=0.5
)
plt.colorbar(scatter, ax=axes[0], label='Lift')
axes[0].axhline(y=min_confianca, color=MINERED, linestyle='--', alpha=0.5,
                label=f'Min Confianca={min_confianca}')
axes[0].axvline(x=min_suporte, color=MINEBLUE, linestyle='--', alpha=0.5,
                label=f'Min Suporte={min_suporte}')
# Rotular top regras
top_regras = df_regras.nlargest(5, 'lift')
for _, row in top_regras.iterrows():
    axes[0].annotate(f'{row["antecedente"]}\n->{row["consequente"]}',
                     (row['suporte'], row['confianca']),
                     fontsize=6, ha='center', textcoords='offset points',
                     xytext=(0, 8))
axes[0].set_xlabel('Suporte')
axes[0].set_ylabel('Confianca')
axes[0].set_title('Bubble Chart - Regras de Associacao\n(tamanho = lift)')
axes[0].legend(fontsize=7)

# 2. Heatmap de co-ocorrencia
co_matrix = pd.DataFrame(0.0, index=produtos, columns=produtos)
for (p1, p2), val in co_ocorrencia.items():
    co_matrix.loc[p1, p2] = val
    co_matrix.loc[p2, p1] = val
np.fill_diagonal(co_matrix.values,
                  [suporte_item[p] for p in produtos])
mask_diag = np.eye(len(produtos), dtype=bool)
sns.heatmap(co_matrix, annot=True, fmt='.2f', cmap='Blues',
            ax=axes[1], linewidths=0.3,
            annot_kws={'size': 7})
axes[1].set_title('Heatmap de Co-ocorrencia de Itens')
axes[1].tick_params(axis='x', rotation=45)

# 3. Grafo de regras de associacao
G_assoc = nx.DiGraph()
for prod in produtos:
    G_assoc.add_node(prod)
top_regras_graf = df_regras.nlargest(15, 'lift')
for _, row in top_regras_graf.iterrows():
    G_assoc.add_edge(row['antecedente'], row['consequente'],
                     weight=row['lift'], confianca=row['confianca'])
pos_assoc = nx.spring_layout(G_assoc, seed=42, k=2)
edge_widths = [G_assoc[u][v]['weight'] for u, v in G_assoc.edges()]
edge_colors = [G_assoc[u][v]['confianca'] for u, v in G_assoc.edges()]
node_sizes_assoc = [suporte_item[n] * 4000 + 300 for n in G_assoc.nodes()]
nx.draw_networkx(
    G_assoc, pos=pos_assoc, ax=axes[2],
    node_size=node_sizes_assoc, node_color=MINEBLUE, alpha=0.9,
    font_size=7, font_color='white', font_weight='bold',
    edge_color=edge_colors, edge_cmap=plt.cm.YlOrRd,
    width=np.array(edge_widths) * 2,
    arrows=True, arrowsize=15,
    connectionstyle='arc3,rad=0.1'
)
axes[2].set_title('Grafo de Regras de Associacao\n(espessura=lift, cor=confianca)')
axes[2].axis('off')

plt.suptitle('Visualizacao de Regras de Associacao - Dados de Mercado',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('regras_associacao.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Auto-EDA com ydata-profiling

Geramos um relatorio automatico de EDA para um dataset com multiplos tipos de dados.

In [ ]:
# Dataset misto para Auto-EDA
np.random.seed(42)
n_rows = 800
df_eda = pd.DataFrame({
    'idade': np.random.randint(18, 75, n_rows),
    'renda_mensal': np.random.lognormal(8.5, 0.8, n_rows),
    'score_credito': np.clip(np.random.normal(650, 120, n_rows), 300, 850),
    'num_produtos': np.random.poisson(2.5, n_rows),
    'dias_cliente': np.random.exponential(730, n_rows).astype(int),
    'num_transacoes': np.random.negative_binomial(5, 0.3, n_rows),
    'estado': np.random.choice(['PA','SP','RJ','MG','RS','BA','PR'], n_rows,
                                p=[0.1,0.25,0.2,0.15,0.1,0.1,0.1]),
    'genero': np.random.choice(['M','F','Outro'], n_rows, p=[0.48,0.48,0.04]),
    'segmento': np.random.choice(['Bronze','Prata','Ouro','Platina'], n_rows,
                                  p=[0.4,0.3,0.2,0.1]),
    'churn': np.random.choice([0, 1], n_rows, p=[0.85, 0.15])
})

# Introducir valores ausentes e anomalias
mask_na = np.random.choice(n_rows, 50, replace=False)
df_eda.loc[mask_na[:25], 'renda_mensal'] = np.nan
df_eda.loc[mask_na[25:], 'score_credito'] = np.nan
# Outliers extremos
df_eda.loc[np.random.choice(n_rows, 5), 'renda_mensal'] = 500000

print(f'Dataset criado: {df_eda.shape}')
print(f'Valores ausentes:\n{df_eda.isnull().sum()}')

# Tentar ydata-profiling
try:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(df_eda, title='EDA Automatico - Dataset de Clientes',
                            explorative=True, minimal=False)
    profile.to_file('eda_report.html')
    print('\nRelatorio EDA salvo em: eda_report.html')
    print('Abra o arquivo no navegador para explorar o relatorio interativo.')

    # Extrair informacoes do relatorio
    descricao = profile.description_set
    print(f'\nAlertas detectados: {len(descricao.alerts)}')
    for alerta in descricao.alerts[:10]:
        print(f'  [{alerta.alert_type.name}] {alerta.column_name}: {alerta}')

except ImportError:
    print('ydata-profiling nao disponivel. Usando pandas_profiling alternativo...')
    try:
        from pandas_profiling import ProfileReport
        profile = ProfileReport(df_eda, title='EDA Automatico', minimal=True)
        profile.to_file('eda_report.html')
        print('Relatorio salvo em eda_report.html')
    except ImportError:
        print('Realizando EDA manual como alternativa:')
        print(df_eda.describe().round(2))
        print('\nCorrelacoes numericas:')
        print(df_eda.select_dtypes(include='number').corr().round(3))

# Analise manual complementar
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
df_eda['renda_mensal'].dropna().hist(bins=30, ax=axes[0,0], color=MINEBLUE, alpha=0.7)
axes[0,0].set_title('Distribuicao de Renda Mensal')
df_eda['score_credito'].dropna().hist(bins=30, ax=axes[0,1], color=MINEORANGE, alpha=0.7)
axes[0,1].set_title('Distribuicao Score de Credito')
df_eda['estado'].value_counts().plot(kind='bar', ax=axes[0,2], color=MINEGREEN, alpha=0.8)
axes[0,2].set_title('Distribuicao por Estado')
axes[0,2].tick_params(axis='x', rotation=45)
df_eda.groupby('segmento')['renda_mensal'].median().plot(kind='bar',
    ax=axes[1,0], color=PALETTE_CB[:4], alpha=0.8)
axes[1,0].set_title('Renda Mediana por Segmento')
df_eda.groupby('churn')['score_credito'].hist(
    bins=20, alpha=0.6, ax=axes[1,1],
    label=['Nao Churn','Churn'],
    color=[MINEBLUE, MINERED])
axes[1,1].set_title('Score de Credito por Churn')
axes[1,1].legend()
corr_num = df_eda.select_dtypes(include='number').corr()
sns.heatmap(corr_num, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=axes[1,2], center=0, cbar=False)
axes[1,2].set_title('Correlacoes')
plt.suptitle('EDA Manual - Dataset de Clientes', fontsize=14)
plt.tight_layout()
plt.savefig('eda_manual.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Mini-Dashboard com Plotly Subplots

Dashboard completo e exportavel como HTML com subplots, anotacoes e layout profissional.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Dados para o dashboard
modelos_comp = ['Logistic Reg.', 'Random Forest', 'XGBoost', 'SVM', 'Neural Net']
acuracias = [0.873, 0.941, 0.956, 0.921, 0.948]
f1_scores = [0.861, 0.933, 0.950, 0.915, 0.942]

# Features e importancias
top_features = [f'Feature_{i}' for i in range(1, 11)]
importancias_dash = np.random.dirichlet(np.ones(10) * 2)
importancias_dash = np.sort(importancias_dash)

# ROC curve (sintetica para dashboard)
fpr_d = np.linspace(0, 1, 100)
tpr_d = np.sqrt(fpr_d) * 0.97 + np.random.uniform(-0.01, 0.01, 100)
tpr_d = np.clip(tpr_d, 0, 1)

# Confusion matrix sintetica
cm_dash = np.array([[85, 8], [5, 116]])

# Curva de aprendizado sintetica
sizes_dash = [50, 100, 200, 300, 400, 455]
train_acc_dash = [0.95, 0.96, 0.965, 0.965, 0.968, 0.970]
val_acc_dash = [0.78, 0.86, 0.91, 0.93, 0.948, 0.956]

# Criar dashboard com subplots
fig_dash = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'Comparacao de Modelos (Acuracia e F1)',
        'Curva ROC (Melhor Modelo - XGBoost)',
        'Top-10 Features Importantes',
        'Matriz de Confusao (XGBoost)',
        'Curvas de Aprendizado',
        'Distribuicao de Scores (Validacao)'
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.12
)

# 1. Comparacao de modelos
fig_dash.add_trace(
    go.Bar(name='Acuracia', x=modelos_comp, y=acuracias,
           marker_color='#0072B2', opacity=0.85,
           text=[f'{v:.1%}' for v in acuracias], textposition='outside'),
    row=1, col=1
)
fig_dash.add_trace(
    go.Bar(name='F1-Score', x=modelos_comp, y=f1_scores,
           marker_color='#E69F00', opacity=0.85,
           text=[f'{v:.1%}' for v in f1_scores], textposition='outside'),
    row=1, col=1
)

# 2. ROC Curve
auc_val = np.trapz(tpr_d, fpr_d)
fig_dash.add_trace(
    go.Scatter(x=fpr_d, y=tpr_d, mode='lines', name=f'ROC (AUC={auc_val:.3f})',
               line=dict(color='#0072B2', width=2.5),
               fill='tozeroy', fillcolor='rgba(0,114,178,0.1)'),
    row=1, col=2
)
fig_dash.add_trace(
    go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Aleatorio',
               line=dict(color='gray', width=1, dash='dash')),
    row=1, col=2
)

# 3. Feature Importance
fig_dash.add_trace(
    go.Bar(x=importancias_dash, y=top_features, orientation='h',
           marker_color='#009E73', name='Importancia',
           text=[f'{v:.3f}' for v in importancias_dash], textposition='outside'),
    row=2, col=1
)

# 4. Confusion Matrix
fig_dash.add_trace(
    go.Heatmap(z=cm_dash,
               x=['Pred: Neg', 'Pred: Pos'],
               y=['Real: Neg', 'Real: Pos'],
               colorscale='Blues',
               text=cm_dash, texttemplate='%{text}',
               showscale=False, name='Conf. Matrix'),
    row=2, col=2
)

# 5. Learning Curves
fig_dash.add_trace(
    go.Scatter(x=sizes_dash, y=train_acc_dash, mode='lines+markers',
               name='Treino', line=dict(color='#0072B2')),
    row=3, col=1
)
fig_dash.add_trace(
    go.Scatter(x=sizes_dash, y=val_acc_dash, mode='lines+markers',
               name='Validacao', line=dict(color='#E69F00', dash='dash')),
    row=3, col=1
)

# 6. Distribuicao de scores
scores_benigno = np.random.beta(8, 2, 300)
scores_maligno = np.random.beta(2, 8, 200)
fig_dash.add_trace(
    go.Histogram(x=scores_benigno, name='Benigno', opacity=0.7,
                 marker_color='#009E73', nbinsx=30),
    row=3, col=2
)
fig_dash.add_trace(
    go.Histogram(x=scores_maligno, name='Maligno', opacity=0.7,
                 marker_color='#D55E00', nbinsx=30),
    row=3, col=2
)

# Layout global
fig_dash.update_layout(
    title=dict(
        text='Dashboard Analitico de Mineracao de Dados | UEPA - Modulo 8',
        font=dict(size=18, color='#0072B2'),
        x=0.5
    ),
    height=950,
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=-0.08, x=0.5, xanchor='center'),
    barmode='group',
    paper_bgcolor='white',
    plot_bgcolor='rgba(245,245,255,0.5)',
    font=dict(family='Arial', size=11)
)

# Anotacoes de KPI
fig_dash.add_annotation(
    text=f'<b>Melhor Acuracia:</b> {max(acuracias):.1%} (XGBoost)',
    xref='paper', yref='paper', x=0.02, y=1.05,
    showarrow=False, font=dict(size=12, color='#0072B2'),
    bgcolor='rgba(0,114,178,0.1)', bordercolor='#0072B2', borderwidth=1
)

fig_dash.update_yaxes(title_text='Score', row=1, col=1, range=[0.7, 1.05])
fig_dash.update_xaxes(title_text='FPR', row=1, col=2)
fig_dash.update_yaxes(title_text='TPR', row=1, col=2)
fig_dash.update_xaxes(title_text='Importancia', row=2, col=1)
fig_dash.update_xaxes(title_text='Tamanho do Treino', row=3, col=1)
fig_dash.update_yaxes(title_text='Acuracia', row=3, col=1)
fig_dash.update_xaxes(title_text='Score Previsto', row=3, col=2)

fig_dash.show()

# Salvar como HTML
fig_dash.write_html('dashboard_mineracao.html')
print('Dashboard salvo em: dashboard_mineracao.html')

## 10. Exercicios

Resolva os exercicios abaixo para consolidar o aprendizado do Modulo 8.

In [ ]:
# =============================================================
# EXERCICIO 1: Quarteto de Anscombe Extendido
# =============================================================
# Objetivo: Criar e visualizar um 5o dataset (Dataset V) para o quarteto
#
# Instrucoes:
# 1. Crie um Dataset V: dado circulo com ruido (use sen/cos)
#    x5 = cos(theta) + ruido, y5 = sin(theta) + ruido
# 2. Calcule as mesmas estatisticas dos outros 4 datasets
#    (media, variancia, correlacao, regressao linear)
# 3. Adicione-o ao grid 3x2 com os 4 originais
# 4. Discuta: a estatistica r captura a relacao circular?

print('EXERCICIO 1: Dataset V - Relacao Circular')
print('Dica: theta = np.linspace(0, 2*np.pi, 11)')
print('x5 = np.cos(theta) * 3 + 9 + ruido')
print('y5 = np.sin(theta) * 3 + 7.5 + ruido')
print('Calcule r e compare com os outros 4 datasets')


# =============================================================
# EXERCICIO 2: Grafico Acessivel a Daltonicos
# =============================================================
# Objetivo: Transformar um grafico com problemas de acessibilidade
#
# Instrucoes:
# 1. Crie um grafico de linhas com 4 series usando cores problematicas
#    (vermelho e verde juntos)
# 2. Crie a versao acessivel com paleta colorblind-safe
#    (use PALETTE_CB definido no inicio)
# 3. Adicione marcadores diferentes para cada serie (o, s, ^, D)
# 4. Adicione tracejados diferentes para cada serie
# 5. Calcule o contrast ratio entre cor de texto e fundo

print('\nEXERCICIO 2: Visualizacao Acessivel')
print('Ferramenta de simulacao: https://www.color-blindness.com/coblis-color-blindness-simulator/')
print('Paleta sugerida: PALETTE_CB =', PALETTE_CB)
print('Marcadores: [\'o\', \'s\', \'^\', \'D\', \'*\', \'v\', \'P\', \'X\']')


# =============================================================
# EXERCICIO 3: Versao Interativa de Grafico Estatico
# =============================================================
# Objetivo: Converter um grafico matplotlib estatico em Plotly interativo
#
# Instrucoes:
# 1. Pegue o pairplot do Exercicio anterior (ou qualquer grafico matplotlib)
# 2. Crie a versao interativa com plotly.express.scatter_matrix (SPLOM)
#    usando df_iris, colorido por 'species'
# 3. Adicione: (a) hover com informacoes completas, (b) seletor de pares
#    na diagonal (histogram ou kde), (c) opcao de exportar como PNG
# 4. Salve como HTML e compare o tamanho do arquivo com a imagem PNG

print('\nEXERCICIO 3: SPLOM Interativo com Plotly')
print('Documentacao: https://plotly.com/python/splom/')
print("fig = px.scatter_matrix(df_iris, dimensions=[...], color='species')")
print("fig.update_traces(diagonal_visible=False, showupperhalf=False)")
print('Salve com: fig.write_html("splom_interativo.html")')